# 第三方库和 PyPI

## 先看一下模块 5 还存在的问题

1. `/api/analyze` 返回的拼音和情感分数还是写死的占位值，分析是假的。
2. 每次分析结果用完就丢，没有历史记录。

## 去哪里找：PyPI

前端世界中，npm 包存放在 npm registry，`npm install animejs` 会从那里下载包。

Python 世界对应的公共包仓库叫 **PyPI（Python Package Index）**：

~~~text
https://pypi.org
~~~

PyPI 中有大量由社区发布的 Python 包。寻找候选库有几种常见方法：

- 直接到 pypi.org 搜关键词，查看简介、安装命令、版本历史和文档链接。
- 使用搜索引擎，例如搜索“Python 中文拼音库”。
- 把需求描述给 AI，让它推荐几个候选。

这三种方式都只能帮助我们找到**候选**，不能保证候选一定可靠。

## AI 只负责给线索，候选必须自己验证

包名、下载量、GitHub 星数只能提供信号，不能单独证明安全。安装第三方包本质上是在本机运行别人的代码，来源不清楚时不要直接安装。

## 锁定两个候选库

回到文字实验室的两个需求：

| 需求 | 候选库 |
| --- | --- |
| 给中文标注拼音，识别多音字 | `pypinyin` |
| 给中文文本计算情感分数 | `snownlp` |

候选有了，但先不要急着安装。下一步先验证它们是否可靠、是否真的满足需求。

## 验库：靠谱吗？能不能满足需求？

验证分为两层。

### 第一层：靠不靠谱

点进去后打开 Homepage 观察几个快速信号：

- 仓库是否有真实用户和社区关注。
- 最近是否还在更新，是否持续发布版本。
- README、安装方式和许可证是否清楚。
- Issue 中是否有长期无人处理的严重问题。
- PyPI 上的项目主页、源码仓库和维护者信息能否互相对应。

**GitHub 星数**和**最近更新时间**最重要，重要项目还应检查版本、发布者、依赖和安全公告。

### 第二层：能不能满足需求

只有文档提供的功能与需求对得上，候选库才算选对。

## 读 README 和官方文档

在 Python 文档中，经常会看到以 `>>>` 开头的示例：

~~~pycon
>>> from pypinyin import pinyin
>>> pinyin("中心")
[['zhōng'], ['xīn']]
~~~

`>>>` 不是需要复制进 `.py` 文件的代码，它是 Python 交互式解释器的提示符。后面会亲自使用它。

## 安装 pypinyin 和 snownlp

~~~bash
uv add pypinyin snownlp
~~~

## 认识 REPL

安装完成后可以写一个测试脚本，也可以先进入 Python 的 REPL 快速试验。

**REPL** 是 Read-Eval-Print Loop 的缩写，即“读取、求值、打印、循环”。它的体感是：敲一行，立刻执行一行。

使用 uv 进入项目环境中的 Python：

~~~bash
uv run python
~~~

如果已经激活 venv，也可以直接运行：

~~~bash
python
~~~

进入后，终端提示符会变成 `>>>`：

~~~pycon
>>> 1 + 1
2
>>> name = "全栈"
>>> name
'全栈'
>>> print("你好，" + name)
你好，全栈
~~~

只要输入的是值 REPL 就会把值显示出来。

## 在 REPL 中验证 pypinyin

在 README 中找到示例来验证：

~~~pycon
>>> from pypinyin import pinyin, Style  
>>> pinyin("你好")
[['nǐ'], ['hǎo']]
~~~

再验证多音字：

~~~pycon
>>> pinyin("重庆")
[['chóng'], ['qìng']]
>>> pinyin("重要")
[['zhòng'], ['yào']]
~~~

文档还提供了 `heteronym=True`，用于查看一个字的多个候选读音：

~~~pycon
>>> pinyin("中心", heteronym=True)
[['zhōng', 'zhòng'], ['xīn']]
~~~

## 用 lazy_pinyin 得到更适合项目的格式

`pinyin("重庆")` 返回：

~~~python
[["chóng"], ["qìng"]]
~~~

这是嵌套列表，我们不想要

pypinyin 提供了 `lazy_pinyin`：

~~~pycon
>>> from pypinyin import lazy_pinyin
>>> lazy_pinyin("重庆")
['chong', 'qing']
~~~

默认结果不带声调。需要声调时，引入 `Style` 并选择 `Style.TONE`：

~~~pycon
>>> from pypinyin import lazy_pinyin, Style
>>> lazy_pinyin("重庆", style=Style.TONE)
['chóng', 'qìng']
~~~

## 在 REPL 中验证 SnowNLP

导入 `SnowNLP`，分别测试一条偏积极和一条偏消极的文本：

~~~pycon
>>> from snownlp import SnowNLP
>>> SnowNLP("今天的风很轻，适合把想法写下来").sentiments
0.9465...
>>> SnowNLP("太失望了，再也不来了").sentiments
0.0027...
~~~

验证完成后退出 REPL：

~~~pycon
>>> exit()
~~~

## 自然语言处理 NLP 生态

pypinyin 和 snownlp 都属于中文自然语言处理生态。

**NLP（Natural Language Processing，自然语言处理）**，就是让程序处理人类语言的一类技术。

这个生态中还有一个常见的库：`jieba`，中文名常叫“结巴分词”。它能把一句中文切成词语：

~~~text
我来到北京清华大学
→ 我 / 来到 / 北京 / 清华大学
~~~

## 把依赖记上账

### 使用 uv 的项目

执行 `uv add pypinyin snownlp` 后，uv 已经更新：

- `pyproject.toml`：记录项目直接依赖和版本约束。
- `uv.lock`：锁定完整依赖树的精确版本。

这两个文件应该提交到 Git；`.venv/` 不提交。

其他人拿到项目后，可以执行：

~~~bash
uv sync
~~~

uv 会根据 `pyproject.toml` 和 `uv.lock` 重建一致的环境。运行项目命令可以使用：

~~~bash
uv run fastapi dev
~~~